# Logictic Regression

<img src="./images/Logistic_regression_1.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_2.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_3.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_4.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_5.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_6.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_7.png" alt="sample" width="1000"/>

<img src="./images/Logistic_regression_8.png" alt="sample" width="1000"/>

# Binary classification

In [1]:
# Logistic Regression on UCI Wine Quality (binary: good vs not_good)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# --- 1) Load data (red + white) ---
# UCI datasets (CSV, semicolon-separated)
RED_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
WHITE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(RED_URL, sep=';')
white = pd.read_csv(WHITE_URL, sep=';')

# Add a 'type' column so we can combine (optional, not used below)
red["type"] = "red"
white["type"] = "white"

data = pd.concat([red, white], ignore_index=True)

# --- 2) Create binary target: good (>=7) vs not_good (<7) ---
data["target"] = (data["quality"] >= 7).astype(int)  # 1 = good, 0 = not_good

# Features (drop quality and the optional 'type' if included)
X = data.drop(columns=["quality", "target", "type"], errors="ignore")
y = data["target"]

# --- 3) Train/Test split (stratify to preserve class ratio) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# --- 4) Build model pipeline ---
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced",    # handle class imbalance
        solver="liblinear",         # stable for small-ish datasets
        random_state=42,
        max_iter=1000
    ))
])

# --- 5) Train ---
pipe.fit(X_train, y_train)

# --- 6) Predict & evaluate ---
y_pred = pipe.predict(X_test)
# For ROC-AUC we need probabilities of class 1
y_proba = pipe.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_proba)
cm   = confusion_matrix(y_test, y_pred)

print("=== Logistic Regression (Wine Quality) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}\n")

print("Confusion Matrix [tn fp; fn tp]:")
print(cm, "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4, target_names=["not_good", "good"]))

# --- 7) Optional: inspect learned coefficients ---
# Map coefficients back to feature names for interpretability
lr = pipe.named_steps["lr"]
coef = pd.Series(lr.coef_.ravel(), index=X.columns).sort_values(ascending=False)
print("\nTop positive coefficients (push towards 'good'):")
print(coef.head(10))
print("\nTop negative coefficients (push towards 'not_good'):")
print(coef.tail(10))

=== Logistic Regression (Wine Quality) ===
Accuracy : 0.7177
Precision: 0.3888
Recall   : 0.7578
F1-score : 0.5139
ROC-AUC  : 0.8003

Confusion Matrix [tn fp; fn tp]:
[[739 305]
 [ 62 194]] 

Classification Report:
              precision    recall  f1-score   support

    not_good     0.9226    0.7079    0.8011      1044
        good     0.3888    0.7578    0.5139       256

    accuracy                         0.7177      1300
   macro avg     0.6557    0.7328    0.6575      1300
weighted avg     0.8175    0.7177    0.7445      1300


Top positive coefficients (push towards 'good'):
residual sugar          0.906695
alcohol                 0.822536
fixed acidity           0.556579
sulphates               0.364180
pH                      0.312395
free sulfur dioxide     0.182142
citric acid            -0.085102
chlorides              -0.119357
total sulfur dioxide   -0.303641
volatile acidity       -0.734821
dtype: float64

Top negative coefficients (push towards 'not_good'):
alcohol  

/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manojkumar_rajendran/Library/Python/3.13/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = 

# Multi-class classification

In [ ]:
path = r'/Users/manojkumar_rajendran/Desktop/MR/Technology_Learning/TutorNotes/Data_Science/Classical_ML/dataset'
import pandas as pd
df = pd.read_csv(path + '/' + 'WineQT.csv')

y = df['quality']
X = df.drop('Id', axis=1).drop('quality', axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced",    # handle class imbalance
        solver="liblinear",         # stable for small-ish datasets
        random_state=42,
        max_iter=1000
    ))
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)

acc  = accuracy_score(y_test, y_pred)
print(y_proba)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0, average="weighted")
rec  = recall_score(y_test, y_pred, zero_division=0, average="weighted")
f1   = f1_score(y_test, y_pred, zero_division=0, average="weighted")
auc  = roc_auc_score(y_test, y_proba, multi_class='ovo')
cm   = confusion_matrix(y_test, y_pred)

print("=== Logistic Regression (Wine Quality) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}\n")

print("Confusion Matrix [tn fp; fn tp]:")
print(cm, "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Hyperparameters of Logistic Regression - Notes

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced",    # handle class imbalance
        solver="liblinear",         # stable for small-ish datasets
        random_state=42,
        max_iter=1000
    ))
])

Why Scaling?

    Features like alcohol %, acidity, and residual sugar may have different units and ranges.

    Logistic Regression uses weights for each feature → large-scale features would dominate if not standardized.

    Scaling gives fair importance to all features and makes model coefficients comparable.

class_weight="balanced":

    Automatically adjusts weights for classes based on frequency. 
    
    Prevents the model from being biased towards majority classes (like quality 5 or 6). 
    
    Rare quality levels (like 3 or 8) get higher importance in training.

    It does not change your dataset (no oversampling/undersampling).

    It only changes how the model “pays attention” to classes during training.


solver="liblinear" is a stable optimization algorithm.

    Works well for small to medium datasets and supports class weights.

max_iter=1000 allows more iterations for the solver to converge.

random_state=42 fixes randomness (train/test splits, solver starting point). Ensures results are the same every time the code is run.

This pipeline standardizes features, balances class influence, and trains a logistic regression model in a reproducible and stable way. It’s a best practice template for classification tasks with imbalanced datasets and differently scaled features.